# 013 — Parameterised SDOF: backbone fitting and calibration

Derives a parameterised equivalent single-degree-of-freedom (SDOF) model for the 3-storey
EC8-gen2 DC2 concentrically-braced frames, by fitting the MDOF cyclic-pushover response.

The SDOF is two parallel zero-length springs, matching
`phd_project/scripts/templates/template_model_cbf_sdof.py`:

- a **`Steel02`** spring for the soft-storey / column-frame contribution, and
- a **`Hysteretic`** spring for the braced-frame contribution.

Every backbone and hysteresis parameter is predicted from the design base-shear coefficient
`Vb_coeff` (design base shear / total seismic weight) and the braced-bay aspect ratio.

| Notebook | Does |
|---|---|
| `012-mdof_casestudy_analyses` | Build MDOF models + analysis files, design dataset |
| **013** (this one) | Fit the parameterised SDOF to the MDOF response |
| `014-sdof_validation_and_fragilities` | Validate the SDOF by IDA, produce fragility curves |

**Upstream — `012` must have been run, *and* its launchers executed.** This notebook reads:

- `<ROOT>/<tag>/modal/modal_properties.json`, `pushover/po_curve.csv`,
  `cyclic_pushover/po_curve.csv` — and the same three under each `<tag>_ss/`
- `cfg["proc_data"]["dc2_casestudy_dataset"]` — the design dataset from `012` §3

**Downstream:** `014` reads the SDOF parameters and analysis folders written here.

## ⚠ This notebook contains two run barriers

It cannot be executed straight through. Twice you must stop, run a generated launcher, and
come back:

| Section | Then run | Why |
|---|---|---|
| after §5 | `optimisation_3s_steel02_and_hysteretic.py`, `optimisation_3s_mechanism_steel02.py` | §7 reads the optimised hysteresis parameters |
| after §8 | `approx_sdof_3s_cyclic_pushover.py` | §9 plots the approximate-SDOF cyclic pushovers |

Both barriers are marked with a ⛔ heading below.

In [ ]:
%load_ext autoreload
%autoreload 2

## §0 Setup

In [ ]:
import os
import json
import shutil
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
from numpy.polynomial import Polynomial
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TABLEAU_COLORS
from matplotlib.lines import Line2D
from scipy.stats import lognorm
from scipy.optimize import curve_fit
import statsmodels.api as sm

from phd_project.config import config
from phd_project.scripts.standardise_responses import standardise_responses
from standes.analysis.recorders import get_recorder, get_recorders

from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_analysis_config,
    copy_structural_model,
    configure_batch_run_file,
    copy_file,
)

from phd_project.scripts.WP1_ground_motion_set.optimise_sdof_parameters_differential_evolution import (
    validate_optimised_parameters,
)

# Fitting primitives and the *fitted* prediction equations both live in a shared
# module -- single source of truth, also used by 011 and by the SDOF model template.
# See phd_project/scripts/sdof_parameterisation.py
from phd_project.scripts.sdof_parameterisation import (
    linear_model,
    quadratic,
    exponential_shifted,
    linear_constant_model,
    bilinear_piecewise_model,
    calculate_fit_metrics,
    get_approximate_ss_backbone,
    get_approximate_ss_backbone_params,
    get_approximate_br_backbone,
    br_backbone_params,
    get_approximate_hysteretic_hyst_params,
    get_approximate_steel02_hyst_params,
    total_backbone,
)

from fitpo import (
    fit_cbf_backbone,
    fit_envelope,
    fit_piecewise_backbone,
    fit_tetralinear_backbone,
)
from fitpo.fitting import elastic_slope

cfg = config.load_config()

In [ ]:
def _load_pushover_curve(building_folder: Path):
    # load a pushover curve
    po_folder = building_folder / "pushover"
    po_curve_file = po_folder / "po_curve.csv"

    poc = np.loadtxt(po_curve_file, delimiter=",")
    return poc

def _load_cyclic_pushover_curve(building_folder: Path):
    # load a pushover curve
    po_folder = building_folder / "cyclic_pushover"
    po_curve_file = po_folder / "po_curve.csv"

    poc = np.loadtxt(po_curve_file, delimiter=",")
    return poc

In [ ]:
ROOT = cfg["analysis_data"]["dc2_sdof_fitting"]

## §1 Load the MDOF results and convert to equivalent SDOF

Loads the design dataset, then for every 3-storey building loads the monotonic and cyclic
pushover curves for both the as-designed frame and its soft-storey (`_ss`) variant, and
converts them to equivalent-SDOF coordinates using the modal participation factor `gamma`
and effective modal mass `m*`.

The participation factors are written to disk because `014` needs them to map SDOF intensity
measures back into MDOF-equivalent coordinates.

In [ ]:
# load the building dataset
with open(cfg["proc_data"]["dc2_casestudy_dataset"], "rb") as f:
    bdata_df = pickle.load(f)

bdata_df = bdata_df.set_index("name")
# 012 writes the dataset with a clean RangeIndex, so there is no stray "index" column
# to drop here (older pickles produced by the previous 014 notebook did have one).
bdata_df = bdata_df.drop(columns="index", errors="ignore")
bdata_df

storey_filter = 3
buildings = bdata_df[bdata_df["n_storeys"] == storey_filter].index
# buildings = bdata_df.index

pocs = {}

for b in buildings:
    pocs[b] = _load_pushover_curve(ROOT / b)

In [ ]:
# Participation-factor / equivalent-SDOF conversion now lives in a shared module
# (single source of truth, also used by wp1pt4pt1f). See
# phd_project/scripts/equivalent_sdof.py
from phd_project.scripts.equivalent_sdof import (
    _calculate_participation_factor,
    convert_poc_to_eq_sdof_poc,
)

In [ ]:
def cumulative_distance(d):
    """
    Calculates cumulative absolute displacement.
    i.e. the total distance the roof node travelled
    """
    return np.concatenate(([0], np.cumsum(np.abs(np.diff(d)))))

In [ ]:
pocs = {}
pocs_ss = {}
cpocs = {}
cpocs_ss = {}
eq_sdof_cpoc = {}
eq_sdof_cpoc_ss = {}
gammas = {}
m_stars = {}
gammas_ss = {}
for b in buildings:
    if bdata_df.loc[b, "n_storeys"] == storey_filter:
        
        pocs[b] = _load_pushover_curve(ROOT / b)
        pocs_ss[b] = _load_pushover_curve(ROOT / f"{b}_ss")
        cpocs[b] = _load_cyclic_pushover_curve(ROOT / b)
        cpocs_ss[b] = _load_cyclic_pushover_curve(ROOT / f"{b}_ss")

        eq_sdof_cpoc[b], gammas[b], m_stars[b], *_ = convert_poc_to_eq_sdof_poc(ROOT / b / "modal", cpocs[b])

        eq_sdof_cpoc_ss[b], gammas_ss[b], *_ = convert_poc_to_eq_sdof_poc(ROOT / f"{b}_ss" / "modal", cpocs_ss[b])


# save the participation factors
folder = cfg["proc_data"]["sdof_parameters"]
with open(folder / "participation_factors.json", "w") as file:
    json.dump(gammas, file, indent=4)

with open(folder / "participation_factors_mechanism.json", "w") as file:
    json.dump(gammas_ss, file, indent=4)

## §2 Backbone comparisons

We want SDOF systems that have a suitable *cyclic* response. At this stage the static
pushover response is not the concern — the spring backbones are calibrated against what is
observed in the cyclic pushover rather than the monotonic one.

### Compare the 3-storey cyclic pushover curves

In [ ]:
total_n = len(cpocs)
n_rows = int(np.ceil(total_n / 3))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

for (b, cpoc), ax in zip(cpocs.items(), axs.flatten()):
    
    poc = pocs[b]
    # poc_ss = pocs_ss[b]
    cpoc_ss = cpocs_ss[b]
    # bb_ss = eq_sdof_bbs_ss[b] * sdof_conv_data_ss[b]["gamma"]
    ax.plot(cpoc[:, 0], cpoc[:, 1], color="k", label="CPO Full sys.")
    ax.plot(cpoc_ss[:, 0], cpoc_ss[:, 1], color="0.8", ls="-.", label="CPO Soft St.")
    ax.plot(poc[:, 0], poc[:, 1], color="b", label="PO Full sys.")
    # ax.plot(poc_ss[:, 0], poc_ss[:, 1], color="r", label="PO Soft St.")
    # ax.plot(bb_ss[:, 0], bb_ss[:, 1], color="r", ls="--", label="BB Soft St.")

    ax.set_title(b)
    ax.grid(ls="-.", color="0.8")
    
    print(b)
    print(f"    Full System : {cumulative_distance(cpoc[:, 0])[-1]:.2f} mm")
    print(f"    SS System   : {cumulative_distance(cpoc_ss[:, 0])[-1]:.2f} mm\n")

ax.set_xlim(-500, 500)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

The difference between the cyclic response of the full system and the soft-storey system can
be attributed to the braces. That difference is what §4 parameterises.

In [ ]:
# harmonise the cyclic displacements
cpocs_br = {}

for b in buildings:
    cpoc_total, cpoc_ss, common_s = standardise_responses(cpocs[b], cpocs_ss[b])
    cpoc_brace = np.column_stack([cpoc_total[:, 0], cpoc_total[:, 1] - cpoc_ss[:, 1]])
    cpocs_br[b] = cpoc_brace

### §2.1 Five- and seven-storey pushovers — *side comparison*

> **Not part of the main chain.** The parameterisation targets the 3-storey structures only.
> This section is kept as a sanity check on how the response trends with height; nothing
> downstream depends on it, and it can be skipped.

In [ ]:
pocs_57 = {}
pocs_57_ss = {}
cpocs_57 = {}
cpocs_57_ss = {}
eq_sdof_cpoc_57 = {}
eq_sdof_cpoc_57_ss = {}
gammas_57 = {}
m_stars_57 = {}
gammas_57_ss = {}
for b in bdata_df.index:
    if bdata_df.loc[b, "n_storeys"] in [5,7]:
        
        pocs_57[b] = _load_pushover_curve(ROOT / b)
        # pocs_57_ss[b] = _load_pushover_curve(ROOT / f"{b}_ss")
        cpocs_57[b] = _load_cyclic_pushover_curve(ROOT / b)
        cpocs_57_ss[b] = _load_cyclic_pushover_curve(ROOT / f"{b}_ss")

        eq_sdof_cpoc_57[b], gammas_57[b], m_stars_57[b], *_ = convert_poc_to_eq_sdof_poc(ROOT / b / "modal", cpocs_57[b])


In [ ]:
total_n = len(cpocs_57)
n_rows = int(np.ceil(total_n / 3))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

for (b, cpoc), ax in zip(cpocs_57.items(), axs.flatten()):
    
    poc = pocs_57[b]
    # poc_ss = pocs_ss[b]
    cpoc_ss = cpocs_57_ss[b]
    # bb_ss = eq_sdof_bbs_ss[b] * sdof_conv_data_ss[b]["gamma"]
    ax.plot(cpoc[:, 0], cpoc[:, 1], color="k", label="CPO Full sys.")
    ax.plot(cpoc_ss[:, 0], cpoc_ss[:, 1], color="0.8", ls="-.", label="CPO Soft St.")
    # ax.plot(poc[:, 0], poc[:, 1], color="b", label="PO Full sys.")
    # ax.plot(poc_ss[:, 0], poc_ss[:, 1], color="r", label="PO Soft St.")
    # ax.plot(bb_ss[:, 0], bb_ss[:, 1], color="r", ls="--", label="BB Soft St.")

    ax.set_title(b)
    ax.grid(ls="-.", color="0.8")
    
    # print(b)
    # print(f"    Full System : {cumulative_distance(cpoc[:, 0])[-1]:.2f} mm")
    # print(f"    SS System   : {cumulative_distance(cpoc_ss[:, 0])[-1]:.2f} mm\n")

ax.set_xlim(-500, 500)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

## §3 Frame (soft-storey) backbone parameterisation

Fit a simplified trilinear backbone to the column-frame contribution, then regress each of
its parameters — yield force `Fy`, initial stiffness `E0` and post-yield slope `b` — against
`Vb_coeff`.

In [ ]:
# Simplified backbones for the frame contribution
total_n = len(buildings)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

envelopes_ss = {}
bb_envs_ss = {}
bb_params_ss = {}

for b, ax in zip(buildings, axs.flatten()):
    envelope_ss, _ = fit_envelope(cpocs_ss[b]) 
    envelopes_ss[b] = envelope_ss

    bb_env_ss = fit_piecewise_backbone(envelope_ss, mode="bilinear_incl_neg_stiffness")
    ke = bb_env_ss[1, 1] / bb_env_ss[1, 0]
    kh = (bb_env_ss[2, 1] - bb_env_ss[1, 1]) / (bb_env_ss[2, 0] - bb_env_ss[1, 0])
    kh_ke = kh / ke
    dy = bb_env_ss[1, 1] / ke
    du = abs(bb_env_ss[1, 1] / kh)
    mu_ult = (du + dy) / dy

    bb_envs_ss[b] = bb_env_ss

    print(f"{b} : fy = {bb_env_ss[1, 1]:.3e}, ke = {ke:.2e}, kh = {kh:.2e}, b = {kh_ke:.2f}, d_u = {du:.2f}, mu_u = {mu_ult:.2f}")
    
    bb_params_ss[b] = {
        "Fy": bb_env_ss[1, 1],
        "E0": ke,
        "b": kh_ke,
        "du": du,
    }

    # poc_ss = pocs_ss[b]
    cpoc_ss = cpocs_ss[b]
    ax.plot(cpoc_ss[:, 0], cpoc_ss[:, 1], color="0.8", ls="-.", label="CPO")
    # ax.plot(poc_ss[:, 0], poc_ss[:, 1], color="r", label="PO SS")
    ax.plot(envelope_ss[:, 0], envelope_ss[:, 1], color="r", ls="--", label="Envelope")
    ax.plot(bb_env_ss[:, 0], bb_env_ss[:, 1], color="b", ls="-.", label="BB")

    ax.set_title(b)
    ax.grid(ls="-.", color="0.8")

ax.set_xlim(-300, 300)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

bb_params_ss_df = pd.DataFrame(bb_params_ss).T

### 2.3.1 Parameterisation of spring backbone

In [ ]:
bdata_df["Vb_coeff"] = bdata_df["design_baseshear"] / (bdata_df["seismic_mass"] * 9.81)
x_plot = np.linspace(0, 0.7)
storey_height = 3500
bay_width = 7000
aspect = storey_height / bay_width

### Model for $F_y$

The yield point of the columns is normalised by the design base shear.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

Fy_norm = bb_params_ss_df.loc[buildings, "Fy"] / bdata_df.loc[buildings, "design_baseshear"]
ax.plot(bdata_df.loc[buildings, "Vb_coeff"], Fy_norm,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Piecewise Linear - This is appropriate because there is a physical change 
# between the second and third point. Points 1 and 2 are governed by gravity design, 
# while after point 3 the column strength is governed by seismic design.
initial_guess = [0.1, -12.0, 1.2]
Fy_col_lincon_popt, _ = curve_fit(
linear_constant_model, bdata_df.loc[buildings, "Vb_coeff"], Fy_norm, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *Fy_col_lincon_popt)
ax.plot(x_plot, y_plot, color="b", ls="--", label="Linear Constant")

y_pred = linear_constant_model(bdata_df.loc[buildings, "Vb_coeff"], *Fy_col_lincon_popt)
Fy_col_lincon_fit_metrics = calculate_fit_metrics(Fy_norm, y_pred, num_params=len(Fy_col_lincon_popt))

# Bilinear
initial_guess = [0.1, -12.0, 1.2, -1.0]
Fy_col_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, bdata_df.loc[buildings, "Vb_coeff"], Fy_norm, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *Fy_col_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear")

y_bilin_pred = bilinear_piecewise_model(bdata_df.loc[buildings, "Vb_coeff"], *Fy_col_bilin_popt)
Fy_col_bilin_fit_metrics = calculate_fit_metrics(Fy_norm, y_bilin_pred, num_params=len(Fy_col_bilin_popt))

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Normalised Yield Force, $F_y$ / $V_{b,d}$ [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

print(f"Lincon Fit Coefficients:\nx_knot = {Fy_col_lincon_popt[0]:.4f}\nm = {Fy_col_lincon_popt[1]:.4f}\nc = {Fy_col_lincon_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficients:\nx_knot = {Fy_col_bilin_popt[0]:.4f}\nm1 = {Fy_col_bilin_popt[1]:.4f}\nc1 = {Fy_col_bilin_popt[2]:.4f}\nm2 = {Fy_col_bilin_popt[3]:.4f}\n")

print(pd.DataFrame([Fy_col_lincon_fit_metrics, Fy_col_bilin_fit_metrics], index=["Linear Constant", "Bilinear"]))

In [ ]:
bdata_df["Vb_coeff"]

### Model for $E_0$

The stiffness is made dimensionless by dividing by the yield strength and the storey height.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

# brace_alpha = np.arctan(storey_height / bay_width)
# normalisation_factor_vb = bdata_df.loc[buildings, "design_baseshear"] * np.sin(brace_alpha) * np.cos(brace_alpha) ** 2 / storey_height
# thetay = bb_params_ss_df.loc[buildings, "E0"] / normalisation_factor_vb

thetay = bb_params_ss_df.loc[buildings, "Fy"] / bb_params_ss_df.loc[buildings, "E0"] / storey_height


ax.plot(bdata_df.loc[buildings, "Vb_coeff"], thetay,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Linear
initial_guess = [0.013, -0.03]
thetay_popt, _ = curve_fit(
linear_model, bdata_df.loc[buildings, "Vb_coeff"], thetay, 
p0=initial_guess)
y_plot = linear_model(x_plot, *thetay_popt)
ax.plot(x_plot, y_plot, color="k", ls="--", label="Linear")

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Storey Yield Drift, $\theta_y$ [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

y_pred = linear_model(bdata_df.loc[buildings, "Vb_coeff"], *thetay_popt)
thetay_fit_metrics = calculate_fit_metrics(thetay, y_pred, num_params=len(thetay_popt))

print(f"Linear Fit Coefficients:\nm = {thetay_popt[0]:.4f}\nc = {thetay_popt[1]:.4f}\n")
print("Fit Metrics:")
print(thetay_fit_metrics)

### Model for the post-yield slope $b$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

b_data = bb_params_ss_df.loc[buildings, "b"]
ax.plot(bdata_df.loc[buildings, "Vb_coeff"], b_data,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
ax.grid(ls="-.", color="0.8")

# Fits:
# quadratic fit
initial_guess = [0, 1.0, min(b_data) * 1.1]
b_col_quad_popt, _ = curve_fit(
quadratic, bdata_df.loc[buildings, "Vb_coeff"], b_data, p0=initial_guess
)
y_quad = quadratic(x_plot, *b_col_quad_popt)
ax.plot(x_plot, y_quad, color="r", ls="--", label="Quadratic fit")

y_pred_quad = quadratic(bdata_df.loc[buildings, "Vb_coeff"], *b_col_quad_popt)
b_col_quad_fit_metrics = calculate_fit_metrics(b_data, y_pred_quad, num_params=len(b_col_quad_popt))

# exponential fit
initial_guess = [-2, -1.0, min(b_data) * 1.1]
b_col_exp_popt, _ = curve_fit(
exponential_shifted, bdata_df.loc[buildings, "Vb_coeff"], b_data, p0=initial_guess
)
y_exp = exponential_shifted(x_plot, *b_col_exp_popt)
ax.plot(x_plot, y_exp, color="g", ls="--", label="Exponential fit")

y_pred_exp = exponential_shifted(bdata_df.loc[buildings, "Vb_coeff"], *b_col_exp_popt)
b_col_exp_fit_metrics = calculate_fit_metrics(b_data, y_pred_exp, num_params=len(b_col_exp_popt))

# Bilinear
initial_guess = [0.1, -12.0, 1.2, -1.0]
b_col_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, bdata_df.loc[buildings, "Vb_coeff"], b_data, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *b_col_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear")

y_pred_bilin = bilinear_piecewise_model(bdata_df.loc[buildings, "Vb_coeff"], *b_col_bilin_popt)
b_col_bilin_fit_metrics = calculate_fit_metrics(b_data, y_pred_bilin, num_params=len(b_col_bilin_popt))

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Post-Yield Stiffness Ratio, b [-]")
ax.set_ylim(-1.0, 0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

print(f"Quadratic Fit Coefficients:\na = {b_col_quad_popt[0]:.4f}\nb = {b_col_quad_popt[1]:.4f}\nc = {b_col_quad_popt[2]:.4f}\n")
print(f"Exponential Fit Coefficients:\na = {b_col_exp_popt[0]:.4f}\nb = {b_col_exp_popt[1]:.4f}\nc = {b_col_exp_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficients:\nx_knot = {b_col_bilin_popt[0]:.4f}\nm1 = {b_col_bilin_popt[1]:.4f}\nc1 = {b_col_bilin_popt[2]:.4f}\nm2 = {b_col_bilin_popt[3]:.4f}\n")

print(pd.DataFrame([b_col_quad_fit_metrics, b_col_exp_fit_metrics, b_col_bilin_fit_metrics], index=["Quadratic", "Exponential", "Bilinear"]))

### Selected equations

| Parameter | Form |
|---|---|
| $F_y$ | Linear-constant |
| $E_0$ | Linear (storey yield drift) |
| $b$ | Inverted exponential decay |
| $\mu_{ult}$ | Exponential |

> **Where the equations live.** The fitted prediction equations derived in this section are
> stored in `phd_project/scripts/sdof_parameterisation.py`, not redefined here. That module is
> the single source of truth — it is also used by `011` and by
> `phd_project/scripts/templates/template_model_cbf_sdof.py`. The derivation and the fitted
> coefficients are shown in the cells above; if you refit, update the coefficients in the
> module.

### Compare predictions with the originals

In [ ]:
# Simplified backbones for the frame contribution
total_n = len(buildings)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

appx_bb_ss = {}
appx_bb_params_ss = {}

for b, ax in zip(buildings, axs.flatten()):
    
    vb_coeff = bdata_df.loc[b, "Vb_coeff"]
    Wt = bdata_df.loc[b, "seismic_mass"] * 9.81

    appx_bb_params = get_approximate_ss_backbone_params(vb_coeff, Wt, storey_height)
    appx_bb = get_approximate_ss_backbone(vb_coeff, Wt, storey_height)
    appx_bb_ss[b] = appx_bb
    bb_env_ss = bb_envs_ss[b]
    envelope_ss = envelopes_ss[b]

    appx_bb_params_ss[b] = {        # in sdof coordinates
        "Fy": appx_bb_params[0] / gammas[b],
        "E0": appx_bb_params[1],
        "b": appx_bb_params[2],
        "du": (appx_bb[2, 0] - appx_bb[1, 0]) / gammas[b],
        "mu_ult": appx_bb[2, 0] / appx_bb[1, 0]
    }

    cpoc_ss = cpocs_ss[b]
    ax.plot(cpoc_ss[:, 0], cpoc_ss[:, 1], color="0.8", ls="-.", label="CPO SS.")
    ax.plot(envelope_ss[:, 0], envelope_ss[:, 1], color="r", ls="--", label="Envelope SS")
    ax.plot(bb_env_ss[:, 0], bb_env_ss[:, 1], color="b", ls="-.", label="Opti. BB SS")
    ax.plot(appx_bb[:, 0], appx_bb[:, 1], color="g", ls="-.", label="Appx. BB SS")

    ax.set_title(b)
    ax.grid(ls="-.", color="0.8")

ax.set_xlim(-300, 300)
# ax.set_ylim(-8e5, 8e5)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

bb_params_ss_df = pd.DataFrame(bb_params_ss).T

## §4 Brace backbone parameterisation

Same treatment for the braced-frame contribution, which needs five parameters: `Fy`, `E0`,
post-yield slope `b`, and the fracture and residual ductilities `mu_fr` and `mu_r`.

In [ ]:
# plot the brace response
total_n = len(cpocs)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3 ), sharex=True)

for (b, cpoc), ax in zip(cpocs.items(), axs.flatten()):
    
    cpoc_br = cpocs_br[b]
    ax.plot(cpoc[:, 0], cpoc[:, 1], color="k", label="CPO Full sys.")
    ax.plot(cpoc_br[:, 0], cpoc_br[:, 1], color="0.8", ls="-", label="CPO Br.")

    ax.set_title(b)
    ax.grid(ls="-.", color="0.8")
    
ax.set_xlim(-300, 300)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()


In [ ]:
# Simplified backbones for the brace contribution
total_n = len(buildings)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

collapse_residual = True

envelopes_br = {}
bb_envs_br = {}
bb_params_br = {}
bb_params_br_2 = {}

param_tags_2 = ["Fy", "E0", "b", "mu_fr", "mu_r"]

for b, ax in zip(buildings, axs.flatten()):
    cpoc_br = cpocs_br[b]
    envelope_br, _ = fit_envelope(cpoc_br, ) 
    envelopes_br[b] = envelope_br

    bb_env_br = fit_tetralinear_backbone(
        envelope_br, residual_force_fraction=0.1, collapse_residual=collapse_residual)
    bb_envs_br[b] = bb_env_br

    if collapse_residual:
        bb_params_br[b] = np.concatenate([bb_env_br[1:-1, [1,0]], -bb_env_br[1:-1, [1,0]]]).flatten().tolist()
    else:
        bb_params_br[b] = np.concatenate([bb_env_br[1:, [1,0]], -bb_env_br[1:, [1,0]]]).flatten().tolist()

    ax.plot(cpoc_br[:, 0], cpoc_br[:, 1], color="0.8", ls="-.", label="CPO BR.")
    ax.plot(envelope_br[:, 0], envelope_br[:, 1], color="r", ls="--", label="Envelope BR")
    ax.plot(bb_env_br[:, 0], bb_env_br[:, 1], color="g", ls="-.", label="BB BR")

    ax.set_title(b)
    ax.grid(ls="-.", color="0.8")

    fy = bb_env_br[1, 1]
    dy = bb_env_br[1, 0]
    E0 = bb_env_br[1, 1] / bb_env_br[1, 0]
    kh = (bb_env_br[2, 1] - bb_env_br[1, 1]) / (bb_env_br[2, 0] - bb_env_br[1, 0]) / E0
    mu_fr = bb_env_br[2, 0] / dy
    mu_r = bb_env_br[3, 0] / dy

    bb_params_br_2[b] = [fy, E0, kh, mu_fr, mu_r]
    
ax.set_xlim(-300, 300)
leg = axs.flatten()[0].legend(loc="lower right")
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

bb_params_br_df = pd.DataFrame(bb_params_br_2).T
bb_params_br_df.columns = param_tags_2

In [ ]:
bb_params_br_df

### Model for $F_y$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

Fy_norm = bb_params_br_df.loc[buildings, "Fy"] / bdata_df.loc[buildings, "design_baseshear"]
ax.plot(bdata_df.loc[buildings, "Vb_coeff"], Fy_norm,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Piecewise Linear Constant
initial_guess = [0.1, -12.0, 1.2]
Fy_lincon_popt, _ = curve_fit(
linear_constant_model, bdata_df.loc[buildings, "Vb_coeff"], Fy_norm, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *Fy_lincon_popt)
ax.plot(x_plot, y_plot, color="b", ls="--", label="Linear Const.")

y_lincon_pred = linear_constant_model(bdata_df.loc[buildings, "Vb_coeff"], *Fy_lincon_popt)
Fy_lincon_fit_metrics = calculate_fit_metrics(Fy_norm, y_lincon_pred, num_params=len(Fy_lincon_popt))

# Piecewise Bilinear
initial_guess = [0.1, -12.0, 1.2, -1.0]
Fy_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, bdata_df.loc[buildings, "Vb_coeff"], Fy_norm, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *Fy_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear")

y_bilin_pred = bilinear_piecewise_model(bdata_df.loc[buildings, "Vb_coeff"], *Fy_bilin_popt)
Fy_bilin_fit_metrics = calculate_fit_metrics(Fy_norm, y_bilin_pred, num_params=len(Fy_bilin_popt))

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Normalised Yield Force, $F_y$ / $V_{b,d}$ [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

print(f"Lincon Fit Coefficients:\nx_knot = {Fy_lincon_popt[0]:.4f}\nm = {Fy_lincon_popt[1]:.4f}\nc = {Fy_lincon_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficients:\nx_knot = {Fy_bilin_popt[0]:.4f}\nm1 = {Fy_bilin_popt[1]:.4f}\nc1 = {Fy_bilin_popt[2]:.4f}\nm2 = {Fy_bilin_popt[3]:.4f}\n")

print(pd.DataFrame([Fy_lincon_fit_metrics, Fy_bilin_fit_metrics], index=["Linear Constant", "Bilinear"]))

### Model for $E_0$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 
brace_alpha = np.arctan(storey_height / bay_width)

# normalisation_factor_fy = bb_params_br_df.loc[buildings, "Fy"] * np.sin(brace_alpha) * np.cos(brace_alpha) ** 2 / storey_height
# E0_norm_fy = bb_params_br_df.loc[buildings, "E0"] / normalisation_factor_fy
# ax.plot(bdata_df.loc[buildings, "Vb_coeff"], E0_norm_fy,
#         marker="o", ls="", mfc="m", mec="k", label="Data_Fy_normed" )

normalisation_factor_vb = bdata_df.loc[buildings, "design_baseshear"] * np.sin(brace_alpha) * np.cos(brace_alpha) ** 2 / storey_height
E0_norm_Vb = bb_params_br_df.loc[buildings, "E0"] / normalisation_factor_vb

ax.plot(bdata_df.loc[buildings, "Vb_coeff"], E0_norm_Vb,
        marker="o", ls="", mfc="b", mec="k", label="Data_Vb_norm" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Piecewise Linear Constant
initial_guess = [0.1, -12000, 2000]
E0_norm_Vb_lincon_popt, _ = curve_fit(
linear_constant_model, bdata_df.loc[buildings, "Vb_coeff"], E0_norm_Vb, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *E0_norm_Vb_lincon_popt)
ax.plot(x_plot, y_plot, color="b", ls="--", label="Linear Const.")

y_lincon_pred = linear_constant_model(bdata_df.loc[buildings, "Vb_coeff"], *E0_norm_Vb_lincon_popt)
E0_norm_Vb_lincon_fit_metrics = calculate_fit_metrics(E0_norm_Vb, y_lincon_pred, num_params=len(E0_norm_Vb_lincon_popt))

# Piecewise Bilinear
initial_guess = [0.1, -12000, 2000, -1000.0]
E0_norm_Vb_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, bdata_df.loc[buildings, "Vb_coeff"], E0_norm_Vb, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *E0_norm_Vb_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear")

y_bilin_pred = bilinear_piecewise_model(bdata_df.loc[buildings, "Vb_coeff"], *E0_norm_Vb_bilin_popt)
E0_norm_Vb_bilin_fit_metrics = calculate_fit_metrics(E0_norm_Vb, y_bilin_pred, num_params=len(E0_norm_Vb_bilin_popt))

# exponential fit
initial_guess = [100, 1.0, min(E0_norm_Vb) * 1.1]
E0_exp_popt, _ = curve_fit(
exponential_shifted, bdata_df.loc[buildings, "Vb_coeff"], E0_norm_Vb, p0=initial_guess
)
y_exp = exponential_shifted(x_plot, *E0_exp_popt)
ax.plot(x_plot, y_exp, color="g", ls="--", label="Exponential fit")

y_pred_exp = exponential_shifted(bdata_df.loc[buildings, "Vb_coeff"], *E0_exp_popt)
E0_exp_fit_metrics = calculate_fit_metrics(E0_norm_Vb, y_pred_exp, num_params=len(E0_exp_popt))

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Normalised Brace Stiffness, $\bar{E}_{br}$ [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

print(f"Exponential Fit Coefficients:\na = {E0_exp_popt[0]:.4f}\nb = {E0_exp_popt[1]:.4f}\nc = {E0_exp_popt[2]:.4f}\n")
print(f"Lincon Fit Coefficients:\nx_knot = {E0_norm_Vb_lincon_popt[0]:.4f}\nm = {E0_norm_Vb_lincon_popt[1]:.4f}\nc = {E0_norm_Vb_lincon_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficients:\nx_knot = {E0_norm_Vb_bilin_popt[0]:.4f}\nm1 = {E0_norm_Vb_bilin_popt[1]:.4f}\nc1 = {E0_norm_Vb_bilin_popt[2]:.4f}\nm2 = {E0_norm_Vb_bilin_popt[3]:.4f}\n")

print("Fit Metrics:")
print(pd.DataFrame([E0_exp_fit_metrics, E0_norm_Vb_lincon_fit_metrics, E0_norm_Vb_bilin_fit_metrics], index=["Exponential", "Linear Constant", "Bilinear"]))

### Model for the post-yield slope $b$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

b_data = bb_params_br_df.loc[buildings, "b"]
b_data2 = bb_params_br_df.loc[buildings, "b"].values.tolist()
b_data2.pop(6)
Vb_coeff2 = bdata_df.loc[buildings, "Vb_coeff"].values.tolist()
Vb_coeff2.pop(6)
ax.plot(bdata_df.loc[buildings, "Vb_coeff"], b_data,
        marker="o", ls="", mfc="m", mec="k", label="Outlier" )
ax.plot(Vb_coeff2, b_data2,
        marker="o", ls="", mfc="b", mec="k", label="Data wo Outlier" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Piecewise Linear Constant w Outlier
initial_guess = [0.1, -12.0, 1.2]
b_lincon_popt, _ = curve_fit(
linear_constant_model, bdata_df.loc[buildings, "Vb_coeff"], b_data, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *b_lincon_popt)
ax.plot(x_plot, y_plot, color="b", ls="--", label="Lin. Const. w Outlier")

# Piecewise Linear Constant wo Outlier
initial_guess = [0.1, -12.0, 1.2]
b_lincon_popt_woo, _ = curve_fit(
linear_constant_model, Vb_coeff2, b_data2, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *b_lincon_popt_woo)
ax.plot(x_plot, y_plot, color="b", ls=":", label="Lin. Const. wo Outlier")

# Piecewise Bilinear w Outlier
initial_guess = [0.1, -12.0, 1.2, -1.0]
b_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, bdata_df.loc[buildings, "Vb_coeff"], b_data, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *b_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear w Outlier")

# Piecewise Bilinear wo Outlier
b_bilin_popt_woo, _ = curve_fit(
bilinear_piecewise_model, Vb_coeff2, b_data2, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *b_bilin_popt_woo)
ax.plot(x_plot, y_plot, color="r", ls=":", label="Bilinear wo Outlier")

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Post-Yield Stiffness Ratio, b [-]")
ax.set_ylim(-0.3, 0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

y_pred_lincon_woo = linear_constant_model(np.array(Vb_coeff2), *b_lincon_popt_woo)
b_lincon_fit_metrics_woo = calculate_fit_metrics(np.array(b_data2), y_pred_lincon_woo, num_params=len(b_lincon_popt_woo))

y_pred_lincon = linear_constant_model(bdata_df.loc[buildings, "Vb_coeff"], *b_lincon_popt)
b_lincon_fit_metrics = calculate_fit_metrics(b_data, y_pred_lincon, num_params=len(b_lincon_popt))

y_pred_bilin_woo = bilinear_piecewise_model(np.array(Vb_coeff2), *b_bilin_popt_woo)
b_bilin_fit_metrics_woo = calculate_fit_metrics(np.array(b_data2), y_pred_bilin_woo, num_params=len(b_bilin_popt_woo))

y_pred_bilin = bilinear_piecewise_model(bdata_df.loc[buildings, "Vb_coeff"], *b_bilin_popt)
b_bilin_fit_metrics = calculate_fit_metrics(b_data, y_pred_bilin, num_params=len(b_bilin_popt))

print(f"Linear Constant Fit Coefficients wo Outlier:\nx_knot = {b_lincon_popt_woo[0]:.4f}\nm = {b_lincon_popt_woo[1]:.4f}\nc = {b_lincon_popt_woo[2]:.4f}\n")
print(f"Linear Constant Fit Coefficients w Outlier:\nx_knot = {b_lincon_popt[0]:.4f}\nm = {b_lincon_popt[1]:.4f}\nc = {b_lincon_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficients wo Outlier:\nx_knot = {b_bilin_popt_woo[0]:.4f}\nm1 = {b_bilin_popt_woo[1]:.4f}\nc1 = {b_bilin_popt_woo[2]:.4f}\nm2 = {b_bilin_popt_woo[3]:.4f}\n")
print(f"Bilinear Fit Coefficients w Outlier:\nx_knot = {b_bilin_popt[0]:.4f}\nm1 = {b_bilin_popt[1]:.4f}\nc1 = {b_bilin_popt[2]:.4f}\nm2 = {b_bilin_popt[3]:.4f}\n")

print("Fit Metrics:")
print(pd.DataFrame([b_lincon_fit_metrics_woo, b_lincon_fit_metrics, b_bilin_fit_metrics_woo, b_bilin_fit_metrics], index=["Linear Constant wo Outlier", "Linear Constant w Outlier","Bilinear wo Outlier", "Bilinear w Outlier"]))

### Model for $\mu_{fr}$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

pop_idxs = []
outliers = []
outlier_vb = []
mu_data = bb_params_br_df.loc[buildings, "mu_fr"].values.tolist()
vb_coeffs = bdata_df.loc[buildings, "Vb_coeff"].values.tolist()

for idx in pop_idxs:
    outliers.append(mu_data.pop(idx))
    outlier_vb.append(vb_coeffs.pop(idx))

mu_data = np.array(mu_data)
vb_coeffs = np.array(vb_coeffs)

ax.plot(vb_coeffs, mu_data,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
# ax.plot(outlier_vb, outliers,
#         marker="o", ls="", mfc="m", mec="k", label="Outlier" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Linear Constant
initial_guess = [0.1, -12.0, 1.2]
mu_lincon_popt, _ = curve_fit(
linear_constant_model, vb_coeffs, mu_data, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *mu_lincon_popt)
ax.plot(x_plot, y_plot, color="r", ls="--", label="Linear Const.")

y_pred = linear_constant_model(vb_coeffs, *mu_lincon_popt)
mu_lincon_fit_metrics = calculate_fit_metrics(mu_data, y_pred, num_params=len(mu_lincon_popt))

# Bilinear Piecewise
initial_guess = [0.1, -12.0, 1.2, -1.0]
mu_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, vb_coeffs, mu_data, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *mu_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear wo Outlier")

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Fracture Ductility, $\mu_{fr}$ [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

y_pred = bilinear_piecewise_model(vb_coeffs, *mu_bilin_popt)
mu_bilin_fit_metrics = calculate_fit_metrics(mu_data, y_pred, num_params=len(mu_bilin_popt))

print(f"Linear Constant Fit Coefficient:\nx_knot = {mu_lincon_popt[0]:.4f}\nm1 = {mu_lincon_popt[1]:.4f}\nc1 = {mu_lincon_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficient:\nx_knot = {mu_bilin_popt[0]:.4f}\nm1 = {mu_bilin_popt[1]:.4f}\nc1 = {mu_bilin_popt[2]:.4f}\nm2 = {mu_bilin_popt[3]:.4f}\n")

print("Fit Metrics:")
print(pd.DataFrame([mu_lincon_fit_metrics, mu_bilin_fit_metrics], index=["Linear Constant", "Bilinear"]))

### Model for $\mu_{r}$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

pop_idxs = []
outliers = []
outlier_vb = []
mu_data = bb_params_br_df.loc[buildings, "mu_r"].values.tolist()
vb_coeffs = bdata_df.loc[buildings, "Vb_coeff"].values.tolist()

for idx in pop_idxs:
    outliers.append(mu_data.pop(idx))
    outlier_vb.append(vb_coeffs.pop(idx))

mu_data = np.array(mu_data)
vb_coeffs = np.array(vb_coeffs)

ax.plot(vb_coeffs, mu_data,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
# ax.plot(outlier_vb, outliers,
#         marker="o", ls="", mfc="m", mec="k", label="Outlier" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Linear Constant
initial_guess = [0.1, -12.0, 1.2]
mu_lincon_popt, _ = curve_fit(
linear_constant_model, vb_coeffs, mu_data, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *mu_lincon_popt)
ax.plot(x_plot, y_plot, color="r", ls="--", label="Linear Const.")

y_pred = linear_constant_model(vb_coeffs, *mu_lincon_popt)
mu_lincon_fit_metrics = calculate_fit_metrics(mu_data, y_pred, num_params=len(mu_lincon_popt))

# Bilinear Piecewise
initial_guess = [0.1, -12.0, 1.2, -1.0]
mu_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, vb_coeffs, mu_data, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *mu_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear")

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"Final Ductility, $\mu_{r}$ [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

y_pred = bilinear_piecewise_model(vb_coeffs, *mu_bilin_popt)
mu_bilin_fit_metrics = calculate_fit_metrics(mu_data, y_pred, num_params=len(mu_bilin_popt))

print(f"Linear Constant Fit Coefficient:\nx_knot = {mu_lincon_popt[0]:.4f}\nm1 = {mu_lincon_popt[1]:.4f}\nc1 = {mu_lincon_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficient:\nx_knot = {mu_bilin_popt[0]:.4f}\nm1 = {mu_bilin_popt[1]:.4f}\nc1 = {mu_bilin_popt[2]:.4f}\nm2 = {mu_bilin_popt[3]:.4f}\n")

print("Fit Metrics:")
print(pd.DataFrame([mu_lincon_fit_metrics, mu_bilin_fit_metrics], index=["Linear Constant", "Bilinear"]))

### Selected equations

| Parameter | Form |
|---|---|
| $F_y$ | Bilinear |
| $E_0$ | Exponential decay |
| $b$ | Bilinear (outlier excluded) |
| $\mu_{fr}$ | Bilinear-constant |
| $\mu_{r}$ | Linear-constant |

> **Where the equations live.** The fitted prediction equations derived in this section are
> stored in `phd_project/scripts/sdof_parameterisation.py`, not redefined here. That module is
> the single source of truth — it is also used by `011` and by
> `phd_project/scripts/templates/template_model_cbf_sdof.py`. The derivation and the fitted
> coefficients are shown in the cells above; if you refit, update the coefficients in the
> module.

### Comparison with the originals

In [ ]:
# Simplified backbones for the frame contribution
total_n = len(buildings)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

appx_bb_br = {}
appx_bb_params_br = {}

for b, ax in zip(buildings, axs.flatten()):
    
    vb_coeff = bdata_df.loc[b, "Vb_coeff"]
    Wt = bdata_df.loc[b, "seismic_mass"] * 9.81

    appx_bb = get_approximate_br_backbone(vb_coeff, Wt, storey_height, aspect)
    appx_bb_br[b] = appx_bb

    appx_bb_params_br[b] = (np.concatenate([appx_bb[1:, [1,0]], -appx_bb[1:, [1,0]]]) / gammas[b]
                            ).flatten().tolist()

    brace_angle = np.arctan(aspect)
    print(br_backbone_params(vb_coeff, Wt, storey_height, brace_angle))
    print()

    bb_env_br = bb_envs_br[b]
    envelope_br = envelopes_br[b]

    cpoc_br = cpocs_br[b]
    ax.plot(cpoc_br[:, 0], cpoc_br[:, 1], color="0.8", ls="-.", label="CPO BR.")
    ax.plot(envelope_br[:, 0], envelope_br[:, 1], color="r", ls="--", label="Envelope BR")
    ax.plot(bb_env_br[:, 0], bb_env_br[:, 1], color="b", ls="-.", label="Opti. BB BR")
    ax.plot(appx_bb[:, 0], appx_bb[:, 1], color="g", ls="-.", label="Appx. BB BR")

    ax.set_title(b)
    ax.grid(ls="-.", color="0.8")

    
ax.set_xlim(-300, 300)
leg = axs.flatten()[0].legend(loc="lower right")
frame = leg.get_frame()
frame.set_edgecolor("k")

plt.tight_layout()

## §5 Write the SDOF parameters and set up the optimisation

Writes the fitted backbone parameters in SDOF coordinates (divided by `gamma`), the target
displacement and test-data files, and the parameter bounds — then builds the optimisation
configs and their launchers.

The optimisation is a differential-evolution fit of the *hysteresis* parameters (the backbone
is already fixed by §3–§4) against the measured equivalent-SDOF cyclic pushover.

Settings: `popsize=15`, `cores=50`, `seed=1`, `tol=1e-8`, `mutation=(0.5, 1)`,
`recombination=0.7`, `dU_max=2.0`.

Note the SDOF mass is *period-adjusted* (`Ke_sdof * (T / 2pi)^2`) rather than set to `m*`,
so that the SDOF period matches the MDOF period despite the fitted stiffness differing
slightly.

In [ ]:
### SAVE SOME STUFF FOR USING LATER
### CREATE CONFIGS FOR SDOF FITTING

folder = cfg["proc_data"]["sdof_fitting"]

configs_3s_ss = []
configs_3s = []

popsize = 15
cores = 50
seed = 1
tol = 1e-8
mutation = (0.5, 1)
recombination = 0.7
dU_max = 2.0            # we can make this larger because the analysis is stable

for b in buildings:

    # backbone parameters for the soft storey system
    ss_data = bb_params_ss[b].copy()
    # convert to SDOF coordinates - using the gamma for the full system to make sure that all SDOFs have a consistent conversion
    ss_data["Fy"] /= gammas[b]
    ss_data["du"] /= gammas[b]

    filename = f"{b}_mechanism_sdof_parameters.json"
    fp_ss_sdof_params = folder / "sdof_parameters" / filename
    with open(fp_ss_sdof_params, "w") as file:
        json.dump(ss_data, file, indent=4)

    # backbone paraemeters for the full system - converted to SDOF coordinates using the full system gamma
    steel02_bb_params = [v for v in ss_data.values()]   
    hysteretic_bb_params = (np.array(bb_params_br[b]) / gammas[b]).tolist()

    # ideally we would use the m_star mass of the full system, as this is the 
    # single degree of freedom mass. However, as the stiffness is not exactly
    # the same between the SDOF system and the MDOF system due to the fitting
    # procedure we need to tweak the mass slightly to ensure that the period
    # remains the same.
    Ke_sdof = ss_data["E0"] + bb_params_br[b][0] / bb_params_br[b][1]

    with open(ROOT / b / "modal" / "modal_properties.json") as file:
        modal_props = json.load(file)
    
    period = modal_props["eigenPeriod"][0]
    period_adjusted_sdof_mass = Ke_sdof * (period / (2 * np.pi)) ** 2 

    data = {
        "steel02_bb_params": steel02_bb_params,
        "hysteretic_bb_params": hysteretic_bb_params,
        "mass": period_adjusted_sdof_mass                 
    }
    filename = f"{b}_sdof_parameters.json"
    fp_sdof_params = folder / "sdof_parameters" / filename
    with open(fp_sdof_params, "w") as file:
        json.dump(data, file, indent=4)

    # mdof hysteresis data for the soft storey system converted to equivalent sdof coordinates
    eq_sdof_cpoc_ss_data = cpocs_ss[b] / gammas[b]
    fp_ss_test_data = folder / "test_data" / f"{b}_mechanism_eq_sdof_cpo.csv"
    np.savetxt(fp_ss_test_data, eq_sdof_cpoc_ss_data, delimiter=",")  

    # mdof hysteresis data for the full system converted to equivalent sdof coordinates
    eq_sdof_cpoc_data = cpocs[b] / gammas[b]
    fp_test_data = folder / "test_data" / f"{b}_eq_sdof_cpo.csv"
    np.savetxt(fp_test_data, eq_sdof_cpoc_data, delimiter=",") 

    # displacement data for the cyclic pushover curve of the full system, converted to equivalent sdof coordinates
    fp_target_disps = folder / "target_displacements" / f"{b}_target_displacements.csv"  
    np.savetxt(fp_target_disps, cpocs_ss[b][:, 0] / gammas[b], delimiter=",") 

    # parameter bounds for a1 and a2 of the Steel01 material
    bounds = [(0, 0.5), (1.0, 5.0)]     # bounds for a1 and a2
    fp_steel01_param_bounds = folder / "parameter_bounds" / f"{b}_steel_01_parameter_bounds.csv"
    np.savetxt(fp_steel01_param_bounds, bounds, delimiter=",")  

    # parameter bounds for a1, a2, R0, cR1, cR2 of the Steel02 material
    bounds = [(0, 0.5), (1.0, 5.0), (15, 20), (0.5, 0.95), (0.01, 1.0)]     # bounds for a1 and a2
    fp_steel02_param_bounds = folder / "parameter_bounds" / f"{b}_steel_02_parameter_bounds.csv"
    np.savetxt(fp_steel02_param_bounds, bounds, delimiter=",") 

    # parameter bounds for a1, a2, R0, cR1, cR2 of the Steel02 material and pinch_X and pinch_Y, damage_1, damage_2, and beta for the Hysteretic material
    bounds = [(0, 0.5), (1.0, 6.0), (10, 25), (0.5, 0.95), (0.01, 1.0), (0.01, 1.0), (0.01, 1.0), (0, 1.0), (0, 1.0), (0, 1.0)]     # bounds for a1 and a2
    fp_steel02_hysteretic_param_bounds = folder / "parameter_bounds" / f"{b}_steel02_hysteretic_parameter_bounds.csv"
    np.savetxt(fp_steel02_hysteretic_param_bounds, bounds, delimiter=",") 

    # steel02 optimisation for soft storey systems
    config_ss = {
        "results_folder": cfg["proc_data"]["sdof_optimisation"],
        "result_name": "_mechanism_steel02_optimisation.pickle",
        "building_tag": b,
        "sdof_params": fp_ss_sdof_params,             
        "test_data": fp_ss_test_data,        
        "displacements": fp_target_disps,
        "dU_max": dU_max,
        "param_bounds": fp_steel02_param_bounds,
        "initialise_model_func": "initialise_steel02_model",
        "popsize": popsize,
        "cores": cores,
        "seed": seed,
        "tol": tol,
        "mutation": mutation,
        "recombination": recombination,
    }

    config = {
        "results_folder": cfg["proc_data"]["sdof_optimisation"],
        "result_name": "_steel02_and_hysteretic_optimisation.pickle",
        "building_tag": b,
        "sdof_params": fp_sdof_params,             
        "test_data": fp_test_data,        
        "displacements": fp_target_disps,
        "dU_max": dU_max,
        "param_bounds": fp_steel02_hysteretic_param_bounds,
        "initialise_model_func": "steel02_and_hysteretic",
        "popsize": popsize,
        "cores": cores,
        "seed": seed,
        "tol": tol,
        "mutation": mutation,
        "recombination": recombination,
    }

    if bdata_df.loc[b, "n_storeys"] == 3:
        configs_3s_ss.append(config_ss)
        configs_3s.append(config)


### Write the optimisation launchers

Set `WRITE_OPTIMISATION_BATCH = False` to skip this if the optimisation has already been run
and you do not want to overwrite the existing launchers.

In [ ]:
from phd_project.scripts.templates.copy_templates_to_folders import (
    configure_optimisation_batch_run,
)

WRITE_OPTIMISATION_BATCH = True

if WRITE_OPTIMISATION_BATCH:
    src = cfg["templates"]["group_sdof_optimisation"]
    BATCH_ROOT = Path(cfg["scripts"]["wp1pt4pt1_batch_run"])

    dst = BATCH_ROOT / "optimisation_3s_mechanism_steel02.py"
    configure_optimisation_batch_run(src, dst, configs_3s_ss)
    print(f"wrote {dst.name}: {len(configs_3s_ss)} jobs")

    dst = BATCH_ROOT / "optimisation_3s_steel02_and_hysteretic.py"
    configure_optimisation_batch_run(src, dst, configs_3s)
    print(f"wrote {dst.name}: {len(configs_3s)} jobs")
else:
    print("skipped -- using the existing launchers")

---

# ⛔ RUN BARRIER 1 — run the SDOF optimisation now

From `cfg["scripts"]["wp1pt4pt1_batch_run"]`:

```
python optimisation_3s_steel02_and_hysteretic.py   # the one §7 needs
python optimisation_3s_mechanism_steel02.py        # mechanism (soft-storey) variant
```

This is a differential-evolution optimisation over 10 parameters per building — it is slow.
Results land in `cfg["proc_data"]["sdof_optimisation"]` as
`<tag>_steel02_and_hysteretic_optimisation.pickle` and `<tag>_optimised_parameters.json`.

**Do not continue past this point until it has finished.**

## §6 Optimisation results

Compare the optimised SDOF cyclic pushovers against the MDOF targets.

In [ ]:
# Simplified backbones for the brace contribution
total_n = len(configs_3s)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

collapse_residual =True

optimised_hystersis_params = {}
target_responses = {}
simulated_responses = {}

for config, ax, b in zip(configs_3s, axs.flatten(), buildings):
    # load result
    result_file = config["results_folder"] / f"{config['building_tag']}{config['result_name']}"
    with open(result_file, "rb") as file:
        result = pickle.load(file)

    # process the result and save optimised parameters
    steel02_params = result.x[:5]
    hysteretic_params = result.x[5:]
    optimised_params = {
        "steel02_hyst_params": steel02_params.tolist(),
        "hysteretic_hyst_params": hysteretic_params.tolist()
    }

    optimised_params_file = config["results_folder"] / f"{config['building_tag']}_optimised_parameters.json"
    with open(optimised_params_file, "w") as file:
        json.dump(optimised_params, file, indent=4)

    optimised_hystersis_params[b] = optimised_params

    target_response, sim_response = validate_optimised_parameters(
        config, result
    )

    target_responses[b] = target_response
    simulated_responses[b] = sim_response

    ax.plot(target_response[:, 0], target_response[:, 1], color="0.6", ls="-", label="Target (Eq. SDOF)")
    ax.plot(sim_response[:, 0], sim_response[:, 1], color="b", ls="-.", label="SDOF Model")
    
    
    ax.set_title(config["building_tag"])
    ax.grid(ls="-.", color="0.8")

ax.set_xlim(-300, 300)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

## §7 Hysteresis parameter statistics and parameterisation

The `Steel02` hysteresis parameters turn out to be effectively constant across the building
set, so they are taken as the mean. The `Hysteretic` spring's pinching and damage parameters
do vary systematically with `Vb_coeff` and are regressed against it.

The aim is to draw general conclusions and, where possible, parameterise the hysteretic
parameters. What follows is a simple statistical analysis of the optimised parameters to see
where simplifications can be made.

In [ ]:
steel_param_tags = ["a1", "a2", "R0", "cR1", "cR2"]
hyst_param_tags = ["pinchX", "pinchY", "damage1", "damage2", "beta"]

results_dict = {}
for b, params in optimised_hystersis_params.items():
    steel_params = params["steel02_hyst_params"]
    hyst_params = params["hysteretic_hyst_params"]
    vals = steel_params + hyst_params
    tags = steel_param_tags + hyst_param_tags

    results_dict[b] = {t:v for t, v in zip(tags, vals)}

hyst_param_df = pd.DataFrame.from_dict(results_dict, orient="index")
hyst_param_df


In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(20, 7), sharex=True) 
axs = axs.flatten()


for col, ax in zip(hyst_param_df.columns, axs):
    ax.plot(bdata_df.loc[buildings, "Vb_coeff"], hyst_param_df.loc[buildings, col],
            marker="o", ls="", mfc="b", mec="k" )
    ax.set_title(col)
    ...

In [ ]:
hyst_param_stats = hyst_param_df.describe()

Inspecting the plot above, the following simplifications seem logical:

| Parameter | Simplification |
|---|---|
| `a1` | mean, ~0.14 |
| `a2` | mean, ~3.45 |
| `R0` | mean, ~20.18 |
| `cR1` | mean, ~0.78 |
| `cR2` | mean, ~0.30 |
| `pinchX` | bilinear (or mean?) |
| `pinchY` | linear with `Vb_coeff` |
| `damage1` | linear with `Vb_coeff` |
| `damage2` | 0 — only one case where this is not 0 |
| `beta` | mean, ~0.337 |

### Model for pinchX

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

pinchX = hyst_param_df.loc[buildings, "pinchX"]
ax.plot(bdata_df.loc[buildings, "Vb_coeff"], pinchX,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Piecewise Linear Constant
initial_guess = [0.2, -8, 0.9]
pinchX_lincon_popt, _ = curve_fit(
linear_constant_model, bdata_df.loc[buildings, "Vb_coeff"], pinchX, 
p0=initial_guess
)
y_plot = linear_constant_model(x_plot, *pinchX_lincon_popt)
ax.plot(x_plot, y_plot, color="b", ls="--", label="Linear Const.")

y_lincon_pred = linear_constant_model(bdata_df.loc[buildings, "Vb_coeff"], *pinchX_lincon_popt)
pinchX_lincon_fit_metrics = calculate_fit_metrics(pinchX, y_lincon_pred, num_params=len(pinchX_lincon_popt))

# Piecewise Bilinear
initial_guess = [0.2, -8, 0.9, 0.01]
pinchX_bilin_popt, _ = curve_fit(
bilinear_piecewise_model, bdata_df.loc[buildings, "Vb_coeff"], pinchX, 
p0=initial_guess
)
y_plot = bilinear_piecewise_model(x_plot, *pinchX_bilin_popt)
ax.plot(x_plot, y_plot, color="m", ls="--", label="Bilinear")

y_bilin_pred = bilinear_piecewise_model(bdata_df.loc[buildings, "Vb_coeff"], *pinchX_bilin_popt)
pinchX_bilin_fit_metrics = calculate_fit_metrics(pinchX, y_bilin_pred, num_params=len(pinchX_bilin_popt))

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"PinchX [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

print(f"Lincon Fit Coefficients:\nx_knot = {pinchX_lincon_popt[0]:.4f}\nm = {pinchX_lincon_popt[1]:.4f}\nc = {pinchX_lincon_popt[2]:.4f}\n")
print(f"Bilinear Fit Coefficients:\nx_knot = {pinchX_bilin_popt[0]:.4f}\nm1 = {pinchX_bilin_popt[1]:.4f}\nc1 = {pinchX_bilin_popt[2]:.4f}\nm2 = {pinchX_bilin_popt[3]:.4f}\n")

print(pd.DataFrame([pinchX_lincon_fit_metrics, pinchX_bilin_fit_metrics], index=["Linear Constant", "Bilinear"]))

### Model for pinchY

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

pinchY = hyst_param_df.loc[buildings, "pinchY"]
ax.plot(bdata_df.loc[buildings, "Vb_coeff"], pinchY,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Linear
initial_guess = [8, 0.9]
pinchY_lin_popt, _ = curve_fit(
linear_model, bdata_df.loc[buildings, "Vb_coeff"], pinchY, 
p0=initial_guess
)
y_plot = linear_model(x_plot, *pinchY_lin_popt)
ax.plot(x_plot, y_plot, color="k", ls="--", label="Linear")

y_lin_pred = linear_model(bdata_df.loc[buildings, "Vb_coeff"], *pinchY_lin_popt)
pinchY_lin_fit_metrics = calculate_fit_metrics(pinchY, y_lin_pred, num_params=len(pinchY_lin_popt))

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"PinchY [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

print(f"Linear Fit Coefficients:\nm = {pinchY_lin_popt[0]:.4f}\nc = {pinchY_lin_popt[1]:.4f}\n")

print(pd.DataFrame([pinchY_lin_fit_metrics], index=["Linear"]))

### Model for damage1

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5)) 

dam1 = hyst_param_df.loc[buildings, "damage1"]
ax.plot(bdata_df.loc[buildings, "Vb_coeff"], dam1,
        marker="o", ls="", mfc="b", mec="k", label="Data" )
ax.grid(ls="-.", color="0.8")

# Fits:
# Linear
initial_guess = [8, 0.9]
dam1_lin_popt, _ = curve_fit(
linear_model, bdata_df.loc[buildings, "Vb_coeff"], dam1, 
p0=initial_guess
)
y_plot = linear_model(x_plot, *dam1_lin_popt)
ax.plot(x_plot, y_plot, color="k", ls="--", label="Linear")

y_lin_pred = linear_model(bdata_df.loc[buildings, "Vb_coeff"], *dam1_lin_popt)
dam1_lin_fit_metrics = calculate_fit_metrics(dam1, y_lin_pred, num_params=len(dam1_lin_popt))

ax.set_xlabel(r"Baseshear Coeff., $V_{b,d}$ / $W_t$ [-]")
ax.set_ylabel(r"damage1 [-]")
ax.set_ylim(0)
ax.set_xlim(0, 0.7)

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")

print(f"Linear Fit Coefficients:\nm = {dam1_lin_popt[0]:.4f}\nc = {dam1_lin_popt[1]:.4f}\n")

print(pd.DataFrame([dam1_lin_fit_metrics], index=["Linear"]))

### Prediction equations

> **Where the equations live.** The fitted prediction equations derived in this section are
> stored in `phd_project/scripts/sdof_parameterisation.py`, not redefined here. That module is
> the single source of truth — it is also used by `011` and by
> `phd_project/scripts/templates/template_model_cbf_sdof.py`. The derivation and the fitted
> coefficients are shown in the cells above; if you refit, update the coefficients in the
> module.

> The mean `Steel02` parameters are stored there as the constant `STEEL02_HYST_PARAMS_MEAN`.

In [ ]:
get_approximate_steel02_hyst_params()

In [ ]:
get_approximate_hysteretic_hyst_params(bdata_df.loc[b, "Vb_coeff"])

## §8 Build the approximate-SDOF systems

Writes the fully-predicted ("approximate") SDOF parameters — every value coming from the
prediction equations rather than from a per-building fit — and builds a cyclic-pushover
analysis folder for each, so that §9 can check the prediction against the MDOF response.

In [ ]:
for b in buildings:
    # make the target folder
    folder = cfg["proc_data"]["approx_sdof_systems"]
    folder.mkdir(parents=True, exist_ok=True)

    # approximate backbone parameters for the full system - converted to SDOF coordinates using the full system gamma
    appx_steel02_bb_params = [v for v in appx_bb_params_ss[b].values()][:-1]   
    appx_hysteretic_bb_params = appx_bb_params_br[b]
    
    # ideally we would use the m_star mass of the full system, as this is the 
    # single degree of freedom mass. However, as the stiffness is not exactly
    # the same between the SDOF system and the MDOF system due to the fitting
    # procedure we need to tweak the mass slightly to ensure that the period
    # remains the same.
    Ke_appx_sdof = appx_bb_params_ss[b]["E0"] + appx_bb_params_br[b][0] / appx_bb_params_br[b][1]

    with open(ROOT / b / "modal" / "modal_properties.json") as file:
        modal_props = json.load(file)
    
    period = modal_props["eigenPeriod"][0]
    period_adjusted_sdof_mass = Ke_appx_sdof * (period / (2 * np.pi)) ** 2 

    bb_data = {
        "steel02_bb_params": appx_steel02_bb_params,
        "hysteretic_bb_params": appx_hysteretic_bb_params,
        "mass": period_adjusted_sdof_mass
    }
    filename = f"{b}_appx_sdof_bb_parameters.json"
    fp_bb_params = folder / filename
    with open(fp_bb_params, "w") as file:
        json.dump(bb_data, file, indent=4)

    # approximate hystersis parameters for the full system
    hyst_data = {
        "steel02_hyst_params": get_approximate_steel02_hyst_params(),
        "hysteretic_hyst_params": get_approximate_hysteretic_hyst_params(bdata_df.loc[b, "Vb_coeff"]),
    }
    filename = f"{b}_appx_sdof_hyst_parameters.json"
    fp_hyst_params = folder / filename
    with open(fp_hyst_params, "w") as file:
        json.dump(hyst_data, file, indent=4)

In [ ]:
batch = []

for b in buildings:
    # make the target folder
    folder = ROOT / f"{b}_appx_sdof"
    folder.mkdir(parents=True, exist_ok=True)

    # copy the analysis template
    src_run = cfg["templates"]["run_cyclic_pushover"]
    dst_run = folder / "run_run_cyclic_pushover.py"
    copy_file(src_run, dst_run)

    # now copy the structural model
    src_model = cfg["templates"]["model_cbf_sdof"]
    dst_model = folder / "structural_model.py"

    fp_bb_params = cfg["proc_data"]["approx_sdof_systems"] / f"{b}_appx_sdof_bb_parameters.json"
    with open(fp_bb_params, "r") as file:
        sdof_bb_params = json.load(file)

    fp_hyst_params = cfg["proc_data"]["approx_sdof_systems"] / f"{b}_appx_sdof_hyst_parameters.json"    
    with open(fp_hyst_params, "r") as file:
        sdof_hyst_params = json.load(file)
    
    ops_updates = {
        k:v for k,v in (sdof_bb_params|sdof_hyst_params).items()
    }

    copy_structural_model(src_model, dst_model, ops_updates=ops_updates)

    # now copy the config 
    src_config = cfg["templates"]["config_cyclic_pushover"]
    dst_config = folder / "config_cyclic_pushover.py"

    fp_target_disps = cfg["proc_data"]["sdof_fitting"] / "target_displacements" / f"{b}_target_displacements.csv" 
    target_disps = np.loadtxt(fp_target_disps, delimiter=",") 
    
    config_changes = {
        "ctrl_node": 2,
        "displacements": target_disps.tolist()
        }

    copy_analysis_config(src_config, dst_config, update_config=config_changes)
    
    batch.append({
        "script": dst_run,
        "config": [dst_config]
        })
    
batch_run_src = cfg["templates"]["batch_run"]
batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "approx_sdof_3s_cyclic_pushover.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch)

---

# ⛔ RUN BARRIER 2 — run the approximate-SDOF cyclic pushovers

From `cfg["scripts"]["wp1pt4pt1_batch_run"]`:

```
python approx_sdof_3s_cyclic_pushover.py
```

Needed for the comparison plots in §9 below.

## §9 Verification — MDOF vs optimised SDOF vs approximate SDOF

The payoff plot: how well the fully-predicted SDOF reproduces the MDOF cyclic pushover.

In [ ]:
# Simplified backbones for the brace contribution
total_n = len(configs_3s)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

for config, ax, b in zip(configs_3s, axs.flatten(), buildings):

    # load the approximate sdof response
    folder = ROOT / f"{b}_appx_sdof"
    appx_sdof = np.loadtxt(folder / "cyclic_pushover" / "po_curve.csv", delimiter=",")

    target = target_responses[b]

    target, appx_sdof, common_s = standardise_responses(target, appx_sdof)

    opti_sdof = simulated_responses[b]

    ax.plot(target[:, 0], target[:, 1], color="0.6", ls="-", label="Target (Eq. SDOF)")
    ax.plot(opti_sdof[:, 0], opti_sdof[:, 1], color="b", ls="-.", label="Opt. SDOF Model")
    ax.plot(appx_sdof[:, 0], appx_sdof[:, 1], color="g", ls="-.", label="Appx. SDOF Model")
    
    
    ax.set_title(config["building_tag"])
    ax.grid(ls="-.", color="0.8")

ax.set_xlim(-300, 300)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

---

## Next

Continue in **`014-sdof_validation_and_fragilities.ipynb`**, which runs incremental dynamic
analyses on the MDOF, optimised-SDOF and approximate-SDOF systems and derives the collapse
fragility curves consumed by `017-disagg_imls_for_msa_stripes`.